In [1]:
# ============================================================
# CELL 1 — IMPORTS AND LOCAL PATHS
# ============================================================

from pathlib import Path
import re
import sqlite3
import joblib
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Project root
BASE_DIR = Path.cwd()

# Local files
INTENT_PATH = BASE_DIR / "Intent_Classification" / "intent_classifier.pkl"

SENTIMENT_PATH = (
    BASE_DIR
    / "Sentiment_Classification"
    / "amazon_distilbert"
)

DB_PATH = BASE_DIR / "customer_support.db"

print("Project directory:", BASE_DIR)
print("Intent model:", INTENT_PATH)
print("Sentiment model:", SENTIMENT_PATH)
print("Database:", DB_PATH)

print("\nChecking files...")
print("Intent model exists:", INTENT_PATH.exists())
print("Sentiment model exists:", SENTIMENT_PATH.exists())
print("Database exists:", DB_PATH.exists())

c:\Users\ASUS\Downloads\FogProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project directory: c:\Users\ASUS\Downloads\FogProject
Intent model: c:\Users\ASUS\Downloads\FogProject\Intent_Classification\intent_classifier.pkl
Sentiment model: c:\Users\ASUS\Downloads\FogProject\Sentiment_Classification\amazon_distilbert
Database: c:\Users\ASUS\Downloads\FogProject\customer_support.db

Checking files...
Intent model exists: True
Sentiment model exists: True
Database exists: True


In [8]:
# ============================================================
# CELL 2 — GPU / PYTORCH CHECK
# ============================================================

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA version: 12.8


In [9]:
# ============================================================
# CELL 3 — LOAD INTENT CLASSIFIER
# ============================================================

intent_model = joblib.load(INTENT_PATH)

print("Intent model loaded successfully!")
print("Model type:", type(intent_model))

# Get intent classes
intent_classes = intent_model.named_steps["classifier"].classes_

print("\nNumber of intents:", len(intent_classes))

print("\nAvailable intents:")
for i, intent in enumerate(intent_classes, start=1):
    print(f"{i:2}. {intent}")

Intent model loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>

Number of intents: 27

Available intents:
 1. cancel_order
 2. change_order
 3. change_shipping_address
 4. check_cancellation_fee
 5. check_invoice
 6. check_payment_methods
 7. check_refund_policy
 8. complaint
 9. contact_customer_service
10. contact_human_agent
11. create_account
12. delete_account
13. delivery_options
14. delivery_period
15. edit_account
16. get_invoice
17. get_refund
18. newsletter_subscription
19. payment_issue
20. place_order
21. recover_password
22. registration_problems
23. review
24. set_up_shipping_address
25. switch_account
26. track_order
27. track_refund


In [10]:
# ============================================================
# CELL 4 — LOAD SENTIMENT MODEL
# ============================================================

sentiment_tokenizer = AutoTokenizer.from_pretrained(
    str(SENTIMENT_PATH),
    local_files_only=True
)

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    str(SENTIMENT_PATH),
    local_files_only=True
)

# Move model to GPU if available
sentiment_model = sentiment_model.to(device)

# Evaluation mode
sentiment_model.eval()

print("Sentiment model loaded successfully!")
print("Using:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSentiment labels:")
print(sentiment_model.config.id2label)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 790.23it/s]


Sentiment model loaded successfully!
Using: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Sentiment labels:
{0: 'negative', 1: 'positive'}


In [11]:
# ============================================================
# CELL 5 — PREDICTION FUNCTIONS
# ============================================================

def predict_intent(text):
    """
    Predict the customer-support intent.
    """
    intent = intent_model.predict([text])[0]
    return intent


def predict_sentiment(text):
    """
    Predict sentiment and confidence.
    """

    inputs = sentiment_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Move tensors to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = sentiment_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

    prediction = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0][prediction].item()

    sentiment = sentiment_model.config.id2label[
        prediction
    ]

    return sentiment, confidence

In [12]:
# ============================================================
# CELL 5 — PREDICTION FUNCTIONS
# ============================================================

def predict_intent(text):
    """
    Predict the customer-support intent.
    """
    intent = intent_model.predict([text])[0]
    return intent


def predict_sentiment(text):
    """
    Predict sentiment and confidence.
    """

    inputs = sentiment_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Move tensors to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = sentiment_model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

    prediction = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0][prediction].item()

    sentiment = sentiment_model.config.id2label[
        prediction
    ]

    return sentiment, confidence

In [16]:
# ============================================================
# CELL 6 — TEST ML MODELS
# ============================================================

test_text = "My order arrived damaged and I am very disappointed."

intent = predict_intent(test_text)

sentiment, confidence = predict_sentiment(test_text)

print("Text:", test_text)
print("Intent:", intent)
print("Sentiment:", sentiment)
print("Confidence:", round(confidence, 4))

Text: My order arrived damaged and I am very disappointed.
Intent: recover_password
Sentiment: negative
Confidence: 0.9985


In [17]:
# ============================================================
# CELL 7 — INTENT GUARDRAIL
# ============================================================

def intent_guardrail(text, predicted_intent):

    text_lower = text.lower()

    # Password / login
    if any(phrase in text_lower for phrase in [
        "forgot my password",
        "forgot password",
        "reset my password",
        "can't log in",
        "cannot log in",
        "can't login",
        "cannot login",
        "unable to log in"
    ]):
        return "recover_password"

    # Cancel order
    if any(phrase in text_lower for phrase in [
        "cancel my order",
        "cancel the order",
        "cancel order"
    ]):
        return "cancel_order"

    # Track order
    if any(phrase in text_lower for phrase in [
        "where is my order",
        "track my order",
        "track the order",
        "order status",
        "where's my order"
    ]):
        return "track_order"

    # Refund
    if any(phrase in text_lower for phrase in [
        "want a refund",
        "get a refund",
        "request a refund",
        "refund my order"
    ]):
        return "get_refund"

    # Payment
    if any(phrase in text_lower for phrase in [
        "payment problem",
        "payment issue",
        "payment failed",
        "payment doesn't work",
        "payment does not work"
    ]):
        return "payment_issue"

    # Human agent
    if any(phrase in text_lower for phrase in [
        "talk to a human",
        "speak to a human",
        "human agent",
        "real person",
        "customer service agent"
    ]):
        return "contact_human_agent"

    # Damaged / defective / wrong product
    if any(word in text_lower for word in [
        "damaged",
        "broken",
        "defective",
        "wrong item",
        "wrong product"
    ]):
        return "complaint"

    # General feedback
    feedback_words = [
        "happy",
        "satisfied",
        "great",
        "excellent",
        "love",
        "amazing",
        "terrible",
        "awful",
        "disappointed"
    ]

    if any(word in text_lower for word in feedback_words):

        support_keywords = [
            "password",
            "order",
            "refund",
            "payment",
            "shipping",
            "delivery",
            "invoice",
            "account"
        ]

        if not any(
            word in text_lower
            for word in support_keywords
        ):
            return "general_feedback"

    # Otherwise keep ML prediction
    return predicted_intent

In [18]:
# ============================================================
# CELL 8 — PRIORITY ENGINE
# ============================================================

HIGH_INTENTS = {
    "complaint",
    "payment_issue",
    "get_refund",
    "track_refund",
    "cancel_order",
    "contact_human_agent",
    "contact_customer_service"
}

MEDIUM_INTENTS = {
    "track_order",
    "change_shipping_address",
    "change_order",
    "delivery_period",
    "delivery_options",
    "check_cancellation_fee"
}


def calculate_priority(intent, sentiment):

    if intent in HIGH_INTENTS:
        priority = 3

    elif intent in MEDIUM_INTENTS:
        priority = 2

    else:
        priority = 1

    # Negative sentiment increases priority
    if sentiment == "negative":
        priority = min(priority + 1, 3)

    priority_labels = {
        1: "LOW",
        2: "MEDIUM",
        3: "HIGH"
    }

    return priority_labels[priority]

In [19]:
# ============================================================
# CELL 9 — COMPLETE MESSAGE ANALYSIS
# ============================================================

def analyze_message(text):

    # Original ML prediction
    predicted_intent = predict_intent(text)

    # Guardrail correction
    intent = intent_guardrail(
        text,
        predicted_intent
    )

    # Sentiment
    sentiment, confidence = predict_sentiment(text)

    # Priority
    priority = calculate_priority(
        intent,
        sentiment
    )

    return {
        "text": text,
        "original_intent": predicted_intent,
        "intent": intent,
        "sentiment": sentiment,
        "sentiment_confidence": round(confidence, 4),
        "priority": priority
    }

In [20]:
# ============================================================
# CELL 10 — TEST COMPLETE ANALYSIS
# ============================================================

test_messages = [
    "My order arrived damaged and I am very disappointed.",
    "I am very happy with my purchase!",
    "I forgot my password and cannot log into my account.",
    "I want to cancel my order.",
    "Where is my order?",
    "I have a problem with my payment.",
    "I want to get a refund.",
    "I want to talk to a human agent.",
    "What payment methods do you accept?",
    "How long does delivery take?"
]

for message in test_messages:

    result = analyze_message(message)

    print("=" * 80)
    print("Message:", result["text"])
    print("ML intent:", result["original_intent"])
    print("Final intent:", result["intent"])
    print("Sentiment:", result["sentiment"])
    print("Confidence:", result["sentiment_confidence"])
    print("Priority:", result["priority"])

Message: My order arrived damaged and I am very disappointed.
ML intent: recover_password
Final intent: complaint
Sentiment: negative
Confidence: 0.9985
Priority: HIGH
Message: I am very happy with my purchase!
ML intent: recover_password
Final intent: general_feedback
Sentiment: positive
Confidence: 0.9985
Priority: LOW
Message: I forgot my password and cannot log into my account.
ML intent: recover_password
Final intent: recover_password
Sentiment: negative
Confidence: 0.9966
Priority: MEDIUM
Message: I want to cancel my order.
ML intent: cancel_order
Final intent: cancel_order
Sentiment: negative
Confidence: 0.9932
Priority: HIGH
Message: Where is my order?
ML intent: track_order
Final intent: track_order
Sentiment: negative
Confidence: 0.7938
Priority: HIGH
Message: I have a problem with my payment.
ML intent: payment_issue
Final intent: payment_issue
Sentiment: negative
Confidence: 0.9898
Priority: HIGH
Message: I want to get a refund.
ML intent: get_refund
Final intent: get_refun

In [21]:
# ============================================================
# CELL 11 — RESPONSE ENGINE
# ============================================================

def generate_response(intent):

    responses = {

        "cancel_order":
            "I can help you cancel your order. "
            "Please provide your order ID.",

        "track_order":
            "I can help you track your order. "
            "Please provide your order ID.",

        "get_refund":
            "I can help you with your refund. "
            "Please provide your order ID.",

        "track_refund":
            "I can check the status of your refund. "
            "Please provide your order ID.",

        "payment_issue":
            "I'm sorry you're experiencing a payment issue. "
            "Please tell me what happened or provide your order ID.",

        "check_payment_methods":
            "We support several payment methods. "
            "I can provide the available options.",

        "recover_password":
            "I can help you recover your account password. "
            "Please follow the password reset process.",

        "change_shipping_address":
            "I can help you change your shipping address. "
            "Please provide your order ID.",

        "change_order":
            "I can help you modify your order. "
            "Please provide your order ID and tell me what you "
            "would like to change.",

        "delivery_period":
            "I can help you check the expected delivery period "
            "for your order.",

        "delivery_options":
            "I can provide information about the available "
            "delivery options.",

        "check_cancellation_fee":
            "I can check whether a cancellation fee applies "
            "to your order.",

        "check_refund_policy":
            "I can explain our refund policy and help you "
            "determine whether your order is eligible.",

        "complaint":
            "I'm sorry about the problem with your order. "
            "Please provide your order ID so I can help "
            "resolve the issue.",

        "contact_human_agent":
            "I'll help you get in touch with a human "
            "customer-support agent.",

        "contact_customer_service":
            "I can help you contact customer support.",

        "general_feedback":
            "Thank you for your feedback! "
            "We appreciate hearing from you."
    }

    return responses.get(
        intent,
        "I'm here to help. Could you provide more details "
        "about your request?"
    )

In [22]:
# ============================================================
# CELL 12 — BASIC CHATBOT
# ============================================================

def chatbot(text):

    analysis = analyze_message(text)

    response = generate_response(
        analysis["intent"]
    )

    return {
        **analysis,
        "response": response
    }


messages = [
    "I want to get a refund.",
    "Where is my order?",
    "My order arrived damaged and I am very disappointed.",
    "I forgot my password.",
    "I want to cancel my order.",
    "I am very happy with my purchase!"
]

for message in messages:

    result = chatbot(message)

    print("=" * 80)
    print("USER:", result["text"])
    print("INTENT:", result["intent"])
    print("SENTIMENT:", result["sentiment"])
    print("PRIORITY:", result["priority"])
    print("BOT:", result["response"])

USER: I want to get a refund.
INTENT: get_refund
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you with your refund. Please provide your order ID.
USER: Where is my order?
INTENT: track_order
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you track your order. Please provide your order ID.
USER: My order arrived damaged and I am very disappointed.
INTENT: complaint
SENTIMENT: negative
PRIORITY: HIGH
BOT: I'm sorry about the problem with your order. Please provide your order ID so I can help resolve the issue.
USER: I forgot my password.
INTENT: recover_password
SENTIMENT: negative
PRIORITY: MEDIUM
BOT: I can help you recover your account password. Please follow the password reset process.
USER: I want to cancel my order.
INTENT: cancel_order
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you cancel your order. Please provide your order ID.
USER: I am very happy with my purchase!
INTENT: general_feedback
SENTIMENT: positive
PRIORITY: LOW
BOT: Thank you for your feedback! We

In [23]:
# ============================================================
# CELL 12 — BASIC CHATBOT
# ============================================================

def chatbot(text):

    analysis = analyze_message(text)

    response = generate_response(
        analysis["intent"]
    )

    return {
        **analysis,
        "response": response
    }


messages = [
    "I want to get a refund.",
    "Where is my order?",
    "My order arrived damaged and I am very disappointed.",
    "I forgot my password.",
    "I want to cancel my order.",
    "I am very happy with my purchase!"
]

for message in messages:

    result = chatbot(message)

    print("=" * 80)
    print("USER:", result["text"])
    print("INTENT:", result["intent"])
    print("SENTIMENT:", result["sentiment"])
    print("PRIORITY:", result["priority"])
    print("BOT:", result["response"])

USER: I want to get a refund.
INTENT: get_refund
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you with your refund. Please provide your order ID.
USER: Where is my order?
INTENT: track_order
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you track your order. Please provide your order ID.
USER: My order arrived damaged and I am very disappointed.
INTENT: complaint
SENTIMENT: negative
PRIORITY: HIGH
BOT: I'm sorry about the problem with your order. Please provide your order ID so I can help resolve the issue.
USER: I forgot my password.
INTENT: recover_password
SENTIMENT: negative
PRIORITY: MEDIUM
BOT: I can help you recover your account password. Please follow the password reset process.
USER: I want to cancel my order.
INTENT: cancel_order
SENTIMENT: negative
PRIORITY: HIGH
BOT: I can help you cancel your order. Please provide your order ID.
USER: I am very happy with my purchase!
INTENT: general_feedback
SENTIMENT: positive
PRIORITY: LOW
BOT: Thank you for your feedback! We

In [24]:
# ============================================================
# CELL 13 — CONNECT TO EXISTING SQLITE DATABASE
# ============================================================

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found: {DB_PATH}"
    )

conn = sqlite3.connect(
    str(DB_PATH),
    check_same_thread=False
)

cursor = conn.cursor()

print("Connected to database:")
print(DB_PATH)

Connected to database:
c:\Users\ASUS\Downloads\FogProject\customer_support.db


In [25]:
# ============================================================
# CELL 14 — DATABASE INSPECTION
# ============================================================

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

tables = cursor.fetchall()

print("Tables found:")

for table in tables:
    print("-", table[0])

Tables found:
- customers
- orders
- refunds


In [26]:
# ============================================================
# CELL 15 — DATABASE FUNCTIONS
# ============================================================

def get_order_status(order_id):

    cursor.execute("""
        SELECT
            order_id,
            product,
            status,
            delivery_date,
            amount
        FROM orders
        WHERE order_id = ?
    """, (order_id,))

    order = cursor.fetchone()

    if order is None:
        return None

    return {
        "order_id": order[0],
        "product": order[1],
        "status": order[2],
        "delivery_date": order[3],
        "amount": order[4]
    }


def get_refund_status(order_id):

    cursor.execute("""
        SELECT
            refund_id,
            status,
            amount
        FROM refunds
        WHERE order_id = ?
    """, (order_id,))

    refund = cursor.fetchone()

    if refund is None:
        return None

    return {
        "refund_id": refund[0],
        "status": refund[1],
        "amount": refund[2]
    }

In [27]:
# ============================================================
# CELL 17 — ORDER ID EXTRACTION
# ============================================================

def extract_order_id(text):

    match = re.search(
        r"\bORD\d+\b",
        text.upper()
    )

    if match:
        return match.group()

    return None


print(extract_order_id("Where is ORD001?"))
print(extract_order_id("Can you check order ORD004"))
print(extract_order_id("Hello"))

ORD001
ORD004
None


In [28]:
# ============================================================
# CELL 18 — CONVERSATION STATE
# ============================================================

conversation_state = {
    "waiting_for": None,
    "last_intent": None
}

In [29]:
# ============================================================
# CELL 18 — CONVERSATION STATE
# ============================================================

conversation_state = {
    "waiting_for": None,
    "last_intent": None
}

In [30]:
# ============================================================
# CELL 19 — FINAL CONVERSATIONAL CHATBOT
# ============================================================

def chat(message):

    global conversation_state

    # --------------------------------------------------------
    # CASE 1 — We are waiting for an order ID
    # --------------------------------------------------------

    if conversation_state["waiting_for"] == "order_id":

        order_id = extract_order_id(message)

        if order_id is None:

            return (
                "Please provide a valid order ID, "
                "such as ORD001."
            )

        intent = conversation_state["last_intent"]

        # Reset conversation state
        conversation_state["waiting_for"] = None
        conversation_state["last_intent"] = None

        # ----------------------------------------------------
        # Track order
        # ----------------------------------------------------

        if intent == "track_order":

            order = get_order_status(order_id)

            if order is None:

                return (
                    f"I couldn't find order {order_id}. "
                    "Please check the order ID."
                )

            return (
                f"Your order {order['order_id']} for "
                f"{order['product']} is currently "
                f"{order['status']}. "
                f"Expected delivery: "
                f"{order['delivery_date']}."
            )

        # ----------------------------------------------------
        # Track refund
        # ----------------------------------------------------

        if intent == "track_refund":

            refund = get_refund_status(order_id)

            if refund is None:

                return (
                    f"I couldn't find a refund for "
                    f"order {order_id}."
                )

            return (
                f"Your refund {refund['refund_id']} is "
                f"currently {refund['status']}. "
                f"Amount: ${refund['amount']:.2f}."
            )

    # --------------------------------------------------------
    # CASE 2 — Analyze a new message
    # --------------------------------------------------------

    analysis = analyze_message(message)

    intent = analysis["intent"]

    # --------------------------------------------------------
    # CASE 3 — Track order
    # --------------------------------------------------------

    if intent == "track_order":

        conversation_state["waiting_for"] = "order_id"
        conversation_state["last_intent"] = "track_order"

        # Maybe order ID is already in the same message
        order_id = extract_order_id(message)

        if order_id:

            order = get_order_status(order_id)

            conversation_state["waiting_for"] = None
            conversation_state["last_intent"] = None

            if order is None:

                return (
                    f"I couldn't find order {order_id}. "
                    "Please check the order ID."
                )

            return (
                f"Your order {order['order_id']} for "
                f"{order['product']} is currently "
                f"{order['status']}. "
                f"Expected delivery: "
                f"{order['delivery_date']}."
            )

        return (
            "Sure! Please provide your order ID, "
            "for example ORD001."
        )

    # --------------------------------------------------------
    # CASE 4 — Track refund
    # --------------------------------------------------------

    if intent == "track_refund":

        conversation_state["waiting_for"] = "order_id"
        conversation_state["last_intent"] = "track_refund"

        # Maybe order ID is already in the same message
        order_id = extract_order_id(message)

        if order_id:

            refund = get_refund_status(order_id)

            conversation_state["waiting_for"] = None
            conversation_state["last_intent"] = None

            if refund is None:

                return (
                    f"I couldn't find a refund for "
                    f"order {order_id}."
                )

            return (
                f"Your refund {refund['refund_id']} is "
                f"currently {refund['status']}. "
                f"Amount: ${refund['amount']:.2f}."
            )

        return (
            "Sure! Please provide your order ID "
            "so I can check your refund."
        )

    # --------------------------------------------------------
    # CASE 5 — Normal response
    # --------------------------------------------------------

    return generate_response(intent)

In [31]:
# ============================================================
# CELL 20 — TEST ORDER CONVERSATION
# ============================================================

conversation_state = {
    "waiting_for": None,
    "last_intent": None
}

print("USER: Where is my order?")
print("BOT:", chat("Where is my order?"))

print()

print("USER: ORD001")
print("BOT:", chat("ORD001"))

USER: Where is my order?
BOT: Sure! Please provide your order ID, for example ORD001.

USER: ORD001
BOT: Your order ORD001 for Wireless Headphones is currently shipped. Expected delivery: 2026-09-30.


In [32]:
# ============================================================
# CELL 21 — TEST REFUND CONVERSATION
# ============================================================

conversation_state = {
    "waiting_for": None,
    "last_intent": None
}

print("USER: Can you track my refund?")
print("BOT:", chat("Can you track my refund?"))

print()

print("USER: ORD002")
print("BOT:", chat("ORD002"))

USER: Can you track my refund?
BOT: Sure! Please provide your order ID so I can check your refund.

USER: ORD002
BOT: Your refund REF001 is currently processing. Amount: $39.99.


In [33]:
# ============================================================
# CELL 21 — TEST REFUND CONVERSATION
# ============================================================

conversation_state = {
    "waiting_for": None,
    "last_intent": None
}

print("USER: Can you track my refund?")
print("BOT:", chat("Can you track my refund?"))

print()

print("USER: ORD002")
print("BOT:", chat("ORD002"))

USER: Can you track my refund?
BOT: Sure! Please provide your order ID so I can check your refund.

USER: ORD002
BOT: Your refund REF001 is currently processing. Amount: $39.99.


In [34]:
# ============================================================
# CELL 22 — MULTIPLE END-TO-END TESTS
# ============================================================

test_conversations = [
    [
        "Where is my order?",
        "ORD001"
    ],
    [
        "Can you track my refund?",
        "ORD002"
    ],
    [
        "I want to cancel my order."
    ],
    [
        "I forgot my password."
    ],
    [
        "My payment failed."
    ],
    [
        "My order arrived damaged and I am disappointed."
    ],
    [
        "I want to speak to a human agent."
    ],
    [
        "I am very happy with my purchase!"
    ]
]

for conversation in test_conversations:

    conversation_state = {
        "waiting_for": None,
        "last_intent": None
    }

    print("=" * 80)

    for message in conversation:

        print("USER:", message)

        response = chat(message)

        print("BOT:", response)

        print()

USER: Where is my order?
BOT: Sure! Please provide your order ID, for example ORD001.

USER: ORD001
BOT: Your order ORD001 for Wireless Headphones is currently shipped. Expected delivery: 2026-09-30.

USER: Can you track my refund?
BOT: Sure! Please provide your order ID so I can check your refund.

USER: ORD002
BOT: Your refund REF001 is currently processing. Amount: $39.99.

USER: I want to cancel my order.
BOT: I can help you cancel your order. Please provide your order ID.

USER: I forgot my password.
BOT: I can help you recover your account password. Please follow the password reset process.

USER: My payment failed.
BOT: I'm sorry you're experiencing a payment issue. Please tell me what happened or provide your order ID.

USER: My order arrived damaged and I am disappointed.
BOT: I'm sorry about the problem with your order. Please provide your order ID so I can help resolve the issue.

USER: I want to speak to a human agent.
BOT: I'll help you get in touch with a human customer-s

In [35]:
# ============================================================
# CELL 23 — FINAL PIPELINE
# ============================================================

def run_pipeline(message):

    result = analyze_message(message)

    result["response"] = chat(message)

    return result


result = run_pipeline(
    "My order arrived damaged and I am very disappointed."
)

print("Message:", result["text"])
print("Intent:", result["intent"])
print("Sentiment:", result["sentiment"])
print("Confidence:", result["sentiment_confidence"])
print("Priority:", result["priority"])
print("Response:", result["response"])

Message: My order arrived damaged and I am very disappointed.
Intent: complaint
Sentiment: negative
Confidence: 0.9985
Priority: HIGH
Response: I'm sorry about the problem with your order. Please provide your order ID so I can help resolve the issue.


In [36]:
# ============================================================
# CELL 24 — SYSTEM CHECK
# ============================================================

print("=" * 70)
print("FINAL SYSTEM CHECK")
print("=" * 70)

print("Intent model:", "OK" if INTENT_PATH.exists() else "MISSING")
print("Sentiment model:", "OK" if SENTIMENT_PATH.exists() else "MISSING")
print("Database:", "OK" if DB_PATH.exists() else "MISSING")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nFunctions available:")
print("✓ predict_intent")
print("✓ predict_sentiment")
print("✓ intent_guardrail")
print("✓ calculate_priority")
print("✓ analyze_message")
print("✓ generate_response")
print("✓ get_order_status")
print("✓ get_refund_status")
print("✓ extract_order_id")
print("✓ chat")

print("\n✅ Full local pipeline is ready!")

FINAL SYSTEM CHECK
Intent model: OK
Sentiment model: OK
Database: OK
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Functions available:
✓ predict_intent
✓ predict_sentiment
✓ intent_guardrail
✓ calculate_priority
✓ analyze_message
✓ generate_response
✓ get_order_status
✓ get_refund_status
✓ extract_order_id
✓ chat

✅ Full local pipeline is ready!
